In [119]:
import numpy as np
import pandas as pd
import os
from openai import OpenAI
from tqdm.autonotebook import tqdm

# 1. Compute LLM embedding vectors for each MMLU-Pro and GPQA-family question.

In [85]:
# we're working with "mmlu_pro" vs. ["gpqa_diamond", "gpqa_extended", "gpqa_main"]
# can use any submitted model's ``details" file.
mmlu_pro = load_dataset(
    "open-llm-leaderboard/ibm-granite__granite-7b-base-details",
    name="ibm-granite__granite-7b-base__leaderboard_mmlu_pro",
    split="latest")
mmlu_pro = mmlu_pro.to_dict()["doc"]
mmlu_pro_qs = [mmlu_pro[i]["question"] for i in range(len(mmlu_pro))]

In [100]:
# get the GPQA-family - start with diamond
gpqa_diamond = load_dataset(
    "open-llm-leaderboard/ibm-granite__granite-7b-base-details",
    name="ibm-granite__granite-7b-base__leaderboard_gpqa_diamond",
    split="latest")
gpqa_diamond = gpqa_diamond.to_dict()["doc"]
gpqa_diamond_qs = [gpqa_diamond[i]["Question"] for i in range(len(gpqa_diamond))]

# get gpqa_extended
gpqa_extended = load_dataset(
    "open-llm-leaderboard/ibm-granite__granite-7b-base-details",
    name="ibm-granite__granite-7b-base__leaderboard_gpqa_extended",
    split="latest")
gpqa_extended = gpqa_extended.to_dict()["doc"]
gpqa_extended_qs = [gpqa_extended[i]["Question"] for i in range(len(gpqa_extended))]

# get gpqa_main
gpqa_main = load_dataset(
    "open-llm-leaderboard/ibm-granite__granite-7b-base-details",
    name="ibm-granite__granite-7b-base__leaderboard_gpqa_main",
    split="latest")
gpqa_main = gpqa_main.to_dict()["doc"]
gpqa_main_qs = [gpqa_main[i]["Question"] for i in range(len(gpqa_main))]

# put everything together into GPQA all
gpqa_all_qs = gpqa_diamond_qs + gpqa_extended_qs + gpqa_main_qs

In [145]:
# create the openAI client with our key
client = OpenAI(api_key="YOUR API-KEY HERE")

# set a batch-size + the embedding dimension
BATCH_SIZE, EMBED_DIM = 64, 3072

In [146]:
# generate the MMLU-Pro question embeddings
with open("embeddings/mmlu_pro_embeddings.txt", "a") as file:
    for start in tqdm(range(0, len(mmlu_pro_qs), BATCH_SIZE)):

        # which questions are we working with?
        mmlu_pro_batch_qs = mmlu_pro_qs[start:start+BATCH_SIZE]

        # get the embedding if possible
        try:
            response = client.embeddings.create(model="text-embedding-3-large", input=mmlu_pro_batch_qs)
            embedding_batch = np.atleast_2d(np.array([item.embedding for item in response.data], dtype=float))
            
        except Exception as e:

            # status report + append NaN's
            print(f"Error on batch starting at MMLU-Pro question {start}.")
            embedding_batch = np.full(shape=(len(mmlu_pro_batch_qs), EMBED_DIM), fill_value=np.nan, dtype=float)

        # write to our file
        np.savetxt(file, embedding_batch)

# convert to .npy for storage efficiency + delete the .txt version
E = np.loadtxt("embeddings/mmlu_pro_embeddings.txt")
np.save("embeddings/mmlu_pro_embeddings.npy", E)
os.remove("embeddings/mmlu_pro_embeddings.txt")

In [153]:
# generate the GPQA (all) question embeddings
with open("embeddings/gpqa_all_embeddings.txt", "a") as file:
    for start in tqdm(range(0, len(gpqa_all_qs), BATCH_SIZE)):

        # which questions are we working with?
        gpqa_all_batch_qs = gpqa_all_qs[start:start+BATCH_SIZE]

        # get the embedding if possible
        try:
            response = client.embeddings.create(model="text-embedding-3-large", input=gpqa_all_batch_qs)
            embedding_batch = np.atleast_2d(np.array([item.embedding for item in response.data], dtype=float))
            
        except Exception as e:

            # status report + append NaN's
            print(f"Error on batch starting at GPQA (all) question {start}.")
            embedding_batch = np.full(shape=(len(gpqa_all_batch_qs), EMBED_DIM), fill_value=np.nan, dtype=float)

        # write to our file
        np.savetxt(file, embedding_batch)

# convert to .npy for storage efficiency + delete the .txt version
E = np.loadtxt("embeddings/gpqa_all_embeddings.txt")
np.save("embeddings/gpqa_all_embeddings.npy", E)
os.remove("embeddings/gpqa_all_embeddings.txt")

  0%|          | 0/19 [00:00<?, ?it/s]